In [1]:
!pip -q install huggingface_hub scikit-learn

import pandas as pd
from huggingface_hub import hf_hub_download

ruta = hf_hub_download(
    repo_id="SINAI/ALIA-es-discriminative-stance-detection",
    filename="corpus_stance_detection_ALIA.jsonl",
    repo_type="dataset"
)

df = pd.read_json(ruta, lines=True)

print(df.shape)
display(df.head())

corpus_stance_detection_ALIA.jsonl:   0%|          | 0.00/4.48M [00:00<?, ?B/s]

(3000, 8)


,id,target,description,comment,annotation_1,annotation_2,annotation_3,majority_label
0,1,"""Bajar"" las motos de las aceras.",Esta propuesta es originaria del vecino. [NOMB...,A mi no me molesta tanto que aparquen en la ac...,neutral,against,neutral,neutral
1,2,"""Bajar"" las motos de las aceras.",Esta propuesta es originaria del vecino. [NOMB...,Existe una ordenanza municipal que autoriza y ...,against,against,against,against
2,3,"""Bajar"" las motos de las aceras.","Los motoristas, aparte del ruido a veces intol...",Las motos son la solución para descongestionar...,against,neutral,favor,
3,4,"""Bajar"" las motos de las aceras.","Los motoristas, aparte del ruido a veces intol...",Aquí todo se soluciona con multas e impuestos....,neutral,against,against,against
4,5,ARREGLAR LOS REBAJES DE LOS BORDILLOS PARA FAC...,El mal estado de las calles y aceras ha hecho ...,La accesibilidad no es un concepto de patente ...,favor,favor,favor,favor


In [2]:
# Distribución de clases
print(df["majority_label"].value_counts(dropna=False))

# Longitud de los comentarios
print(df["comment"].str.len().describe())

# Acuerdo entre anotadores
cols = ["annotation_1", "annotation_2", "annotation_3"]
n_distintas = df[cols].nunique(axis=1)

print("Acuerdo total:", (n_distintas == 1).sum())
print("Acuerdo mayoritario:", (n_distintas == 2).sum())
print("Desacuerdo total:", (n_distintas == 3).sum())

majority_label
favor      1780
against     627
neutral     443
            150
Name: count, dtype: int64
count    3000.000000
mean      312.591000
std       183.408644
min        79.000000
25%       159.000000
50%       266.000000
75%       444.000000
max       800.000000
Name: comment, dtype: float64
Acuerdo total: 1539
Acuerdo mayoritario: 1311
Desacuerdo total: 150


In [3]:
from sklearn.model_selection import train_test_split

# Eliminamos filas sin etiqueta mayoritaria
df = df.dropna(subset=["majority_label"]).copy()

# Combinamos propuesta y comentario
df["text"] = (
    "Propuesta: " + df["target"].fillna("") +
    "\nComentario: " + df["comment"].fillna("")
)

train, temp = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["majority_label"]
)

val, test = train_test_split(
    temp,
    test_size=0.50,
    random_state=42,
    stratify=temp["majority_label"]
)

print(len(train), len(val), len(test))

2100 450 450


In [4]:
from google.colab import drive
drive.mount("/content/drive")

import os

carpeta = "/content/drive/MyDrive/TP IA y Neuro/data"
os.makedirs(carpeta, exist_ok=True)

train.to_csv(f"{carpeta}/train.csv", index=False)
val.to_csv(f"{carpeta}/val.csv", index=False)
test.to_csv(f"{carpeta}/test.csv", index=False)

Mounted at /content/drive
